In [1]:
# ===========================================================================
# UA-SPEECH DATA PIPELINE - interactive driver
#
# All logic lives in the src/ package; this notebook only calls it, so the
# notebook and run_pipeline.py cannot drift apart. To change behaviour, edit
# the module, not this notebook.
#
# EDA lives in notebooks/02_feature_analysis.ipynb, not here - this notebook
# builds the manifest (scan, verify, filter, label, split, dataset),
# investigates the VAD/padding root cause behind the MFCC "long silent tail"
# (Stage 9), and audits the three-branch severity architecture's data-
# pipeline needs (Stages 11-16: primary severity LOSO protocol, branch
# bottleneck dimensions, framewise segmental/suprasegmental feature audit,
# F0-validity diagnostics, the two preprocessing profiles side by side, and
# a final end-to-end shape sanity check).
#
#   src/config.py         paths, speaker ground truth, label maps, hyperparams
#   src/extraction.py     .tgz archive extraction
#   src/scanning.py       filename parsing, verification, mic filter, labels
#   src/splits.py         LOSO (detection + primary severity) and legacy
#                          balanced 81-fold (severity, secondary only)
#   src/preprocessing.py  resample, Silero VAD trim, pad, MFCC, the two
#                          explicit VAD profiles (speech-focused / temporal-
#                          preserving), segmental/suprasegmental extraction
#   src/vad.py             Silero VAD wrapper (leading/trailing trim, fallback, stats)
#   src/praat.py           framewise F0/voicing/intensity + formant/HNR extraction
#   src/dataset.py        UASpeechDataset
#   src/models/            deep/acoustic/fusion legacy pathways (notebooks/legacy/)
#                          + the three-branch gated_fusion/segmental_pathway/
#                          suprasegmental_pathway architecture
#   src/losses.py          CORAL, complementarity penalty, gradient-reversal layer
# ===========================================================================

# STAGE 0 - Setup. Put the project root on the import path; autoreload picks up
# edits to src/ without a kernel restart.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.console import print_header, print_kv

config.ensure_directories()

print_header("UA-Speech Dysarthria Pipeline")
print_kv("Project root", config.PROJECT_ROOT)
print_kv("Archive folder", config.ARCHIVE_DIR)
print_kv("Audio folder", config.AUDIO_DIR)
print_kv("Ground truth", f"{len(config.CONTROL_IDS)} controls + "
                        f"{len(config.DYSARTHRIC_IDS)} dysarthric = "
                        f"{len(config.ALL_SPEAKERS)} speakers")
print_kv("Microphone channel", config.TARGET_MIC)
print_kv("Words per speaker", config.WORDS_PER_SPEAKER)


══════════════════════════════════════════════════════════════════════════════
  UA-SPEECH DYSARTHRIA PIPELINE
══════════════════════════════════════════════════════════════════════════════
  Project root ............................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification
  Archive folder .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\raw
  Audio folder ............................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\extracted
  Ground truth ............................ 13 controls + 15 dysarthric = 28 speakers
  Microphone channel ...................... M6
  Words per speaker ....................... 765


In [2]:
# STAGE 1 - Extract the archives.
# Copy UASpeech_original_C.tgz and UASpeech_original_FM.tgz into
# data/raw/ first. Runs once; the extracted audio persists in
# data/extracted/, so skip this cell on later passes.
from src.extraction import extract_all_archives

extract_all_archives()


══════════════════════════════════════════════════════════════════════════════
  DATASET EXTRACTION
══════════════════════════════════════════════════════════════════════════════
  Archive folder .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\raw
  Audio extract folder .................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\extracted
  Corpus docs folder ...................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\uaspeech_corpus_docs
  Extracting .............................. UASpeech_original_C.tgz
    UASpeech_original_C.tgz        100%|███████████████████| 69649/69649 [02:41<00:00, 431.90file/s]
  [ ✓ ] Extracted 69,649 file(s) from UASpeech_original_C.tgz
  Extracting .............................. UASpeech_original_FM.tgz
    UASpeech_original_FM.tgz       100%|███████

2

In [3]:
# STAGE 2 - Scan and verify against the 28-speaker ground truth.
# Filenames parse as <Speaker>_<Block>_<WordCode>_<Mic>.wav. Two rules baked
# into parse_filename, both of which the original exploratory scan got wrong:
#   1. The mic channel is taken POSITIONALLY from the final token. The old
#      logic searched for the first token starting with 'M', which matched male
#      speaker IDs like M01 before ever reaching the real mic token.
#   2. macOS resource-fork duplicates ('._' prefix) are skipped - the archive
#      holds one per real .wav, which doubled the apparent file count.
from src.scanning import scan_audio_files, verify_speakers

df_audio = scan_audio_files()
speakers_ok = verify_speakers(df_audio)

df_audio.head()


══════════════════════════════════════════════════════════════════════════════
  DATASET SCAN
══════════════════════════════════════════════════════════════════════════════
  Scanning folder ......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\extracted
  Valid audio files ....................... 143565
  Skipped files ........................... 0
  Unique speakers ......................... 28

─── Speaker Verification ─────────────────────────────────────────────────────
  Expected speakers ....................... 28
  Found speakers .......................... 28
  Missing ................................. None
  Spurious ................................ None
  [ ✓ ] All 28 ground-truth speakers present, no spurious IDs


,Filename,Speaker_ID,Group,Block,WordCode,Microphone_Channel,Filepath
0,CF02_B1_C10_M2.wav,CF02,Healthy Control,B1,C10,M2,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
1,CF02_B1_C10_M3.wav,CF02,Healthy Control,B1,C10,M3,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
2,CF02_B1_C10_M4.wav,CF02,Healthy Control,B1,C10,M4,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
3,CF02_B1_C10_M5.wav,CF02,Healthy Control,B1,C10,M5,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
4,CF02_B1_C10_M6.wav,CF02,Healthy Control,B1,C10,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...


In [4]:
# STAGE 3 - Filter to microphone channel M6 (base-paper protocol: M6 only, all
# three blocks, all word categories, no word-type filtering).
# Expect 21,420 utterances: 11,475 dysarthric + 9,945 healthy control.
from src.scanning import filter_mic_channel

df_m6 = filter_mic_channel(df_audio)


─── Microphone Filter (M6 only) ──────────────────────────────────────────────
  Total M6 samples ........................ 21420
    Dysarthric Patient .................... 11475
    Healthy Control ....................... 9945
  Missing dysarthric speakers on M6 ....... None
  Missing control speakers on M6 .......... None


In [5]:
# STAGE 4 - Validate WAV headers. A handful of UA-Speech files extracted
# zero-filled (see README "Data verification note") - torchaudio/libsndfile
# can't read them, and they would otherwise crash training partway through a
# LOSO run. Dropped here, before word counts, so the count check reflects
# genuinely usable data.
from src.scanning import validate_wav_headers

df_m6 = validate_wav_headers(df_m6)


─── WAV Header Validation ────────────────────────────────────────────────────
  Files checked ........................... 21420
  Corrupted (dropped) ..................... 0
  [ ✓ ] All files have valid RIFF/WAVE headers


In [6]:
# STAGE 5 - Per-speaker word counts; every speaker should have all 765 words on
# M6. Speakers below that are flagged, NOT dropped: an incomplete speaker is
# still usable, and silently removing one would change the LOSO fold count.
from src.scanning import check_word_counts

word_counts = check_word_counts(df_m6)


─── Per-Speaker Word Counts (target: 765) ────────────────────────────────────
  Speaker_ID
  CF02    765
  CF03    765
  CF04    765
  CF05    765
  CM01    765
  CM04    765
  CM05    765
  CM06    765
  CM08    765
  CM09    765
  CM10    765
  CM12    765
  CM13    765
  F02     765
  F03     765
  F04     765
  F05     765
  M01     765
  M04     765
  M05     765
  M07     765
  M08     765
  M09     765
  M10     765
  M11     765
  M12     765
  M14     765
  M16     765
  [ ✓ ] All speakers complete


In [7]:
# STAGE 6 - Severity labels: four classes (Very Low / Low / Mid / High) for the
# 15 dysarthric speakers; controls get 'N/A (Control)'.
from src.scanning import add_severity_labels

df_m6 = add_severity_labels(df_m6)

df_m6.head()


─── Severity Labels ──────────────────────────────────────────────────────────
  [ ✓ ] No unmapped dysarthric speakers
  Severity
  N/A (Control)    9945
  High             3825
  Very Low         3060
  Low              2295
  Mid              2295


,Filename,Speaker_ID,Group,Block,WordCode,Microphone_Channel,Filepath,Severity
4,CF02_B1_C10_M6.wav,CF02,Healthy Control,B1,C10,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
11,CF02_B1_C11_M6.wav,CF02,Healthy Control,B1,C11,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
18,CF02_B1_C12_M6.wav,CF02,Healthy Control,B1,C12,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
25,CF02_B1_C13_M6.wav,CF02,Healthy Control,B1,C13,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
32,CF02_B1_C14_M6.wav,CF02,Healthy Control,B1,C14,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)


In [8]:
# STAGE 7 - Cross-validation splits.
#   Detection: Leave-One-Speaker-Out over all 28 speakers -> 28 folds.
#   Severity PRIMARY: full-population LOSO over all 15 dysarthric speakers.
#   Severity SECONDARY/LEGACY only: config.DROPPED_FOR_BALANCE excludes
#              M12, M08, M09 to reach 3 per class -> 3^4 = 81 iterations.
#
# NOTE: the base paper gives no explicit exclusion list, so that set of three
# speakers is OUR ASSUMPTION. Confirm with the team before treating any
# severity result as final.
from src.splits import build_severity_folds, get_loso_split, summarize_detection_splits

summarize_detection_splits(df_m6)
severity_folds = build_severity_folds(df_m6)


══════════════════════════════════════════════════════════════════════════════
  DETECTION SPLITS (LEAVE-ONE-SPEAKER-OUT)
══════════════════════════════════════════════════════════════════════════════
  Total LOSO folds ........................ 28

─── Example fold (CF02 held out) ─────────────────────────────────────────────
  Train samples ........................... 20655
  Test samples ............................ 765

══════════════════════════════════════════════════════════════════════════════
  SEVERITY SPLITS (BALANCED LEAVE-ONE-PER-CLASS-OUT)
══════════════════════════════════════════════════════════════════════════════
  Speakers dropped for balance ............ ['M12', 'M08', 'M09']

─── Speakers per severity class ──────────────────────────────────────────────
  High .................................... F05, M10, M14
  Low ..................................... F02, M07, M16
  Mid ..................................... F04, M05, M11
  Very Low ..............................

In [9]:
# STAGE 8 - Build the PyTorch dataset. UASpeechDataset returns the raw waveform
# (Deep Pathway), the 39-dim MFCC tensor (Acoustic Pathway), both labels, and
# the speaker ID - so both pathways train on identical audio and splits.
from src.dataset import UASpeechDataset

dataset = UASpeechDataset(df_m6)
sample = dataset[0]

print_header("Dataset Build")
print_kv("Total samples", len(dataset))
print_kv("Waveform shape", tuple(sample["waveform"].shape))
print_kv("MFCC shape", tuple(sample["mfcc"].shape))
print_kv("Detection label", sample["group_label"].item())
print_kv("Severity label", sample["severity_label"].item())
print_kv("Speaker ID", sample["speaker_id"])

manifest_path = config.OUTPUT_DIR / "m6_manifest.csv"
df_m6.to_csv(manifest_path, index=False)
print_kv("Manifest saved", manifest_path)


══════════════════════════════════════════════════════════════════════════════
  DATASET BUILD
══════════════════════════════════════════════════════════════════════════════
  Total samples ........................... 21420
  Waveform shape .......................... (1, 64000)
  MFCC shape .............................. (1, 39, 401)
  Detection label ......................... 0
  Severity label .......................... -1
  Speaker ID .............................. CF02
  Manifest saved .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv


In [10]:
# STAGE 9 - Root-cause investigation: why does the MFCC plot show a large
# near-constant region after the speech?
#
# Two candidate causes, measured here rather than assumed:
#   1. torchaudio.functional.vad() (the OLD preprocessing step, now replaced by
#      Silero VAD in src/vad.py) only trims LEADING silence - it is a forward
#      energy-ramp detector with no concept of a trailing edge. Trailing
#      silence in the raw recording would survive it untouched.
#   2. Every clip is pad/truncated to a FIXED config.CLIP_SECONDS=4.0s window
#      (config.MAX_SAMPLES=64000 samples / 400 MFCC frames). UA-Speech
#      utterances are single isolated words - if the actual spoken content is
#      much shorter than 4s, most of the window is padding regardless of how
#      good the VAD trim is.
#
# This cell measures both directly on a spread of real files: raw duration,
# resampled duration (should be identical - resampling doesn't crop), VAD
# speech duration/ratio, and what fraction of the fixed 4s window is padding.
import pandas as pd
import torchaudio

from src.preprocessing import _load_resampled, load_and_preprocess_with_stats

sample_files = df_m6.groupby("Speaker_ID", group_keys=False).sample(
    n=3, random_state=config.DEFAULT_SEED)
sample_files = sample_files.sample(min(80, len(sample_files)), random_state=config.DEFAULT_SEED)

records = []
for row in sample_files.itertuples(index=False):
    raw_waveform, raw_sr = torchaudio.load(row.Filepath)
    raw_duration_s = raw_waveform.shape[1] / raw_sr

    resampled_waveform, resampled_sr = _load_resampled(row.Filepath)
    resampled_duration_s = resampled_waveform.shape[1] / resampled_sr

    _, _, vad_stats = load_and_preprocess_with_stats(row.Filepath)

    records.append({
        "Filename": row.Filename, "Speaker_ID": row.Speaker_ID, "Group": row.Group,
        "raw_duration_s": raw_duration_s, "resampled_duration_s": resampled_duration_s,
        **vad_stats,
    })

root_cause_df = pd.DataFrame(records)
max_window_s = config.MAX_SAMPLES / config.TARGET_SR
root_cause_df["padding_fraction_of_window"] = (
    1 - root_cause_df["speech_duration_s"] / max_window_s).clip(lower=0)

print_header("Root-cause investigation — MFCC trailing padding")
print_kv("Sample size", len(root_cause_df))
print_kv("Fixed analysis window", f"{max_window_s:.1f}s ({config.MAX_SAMPLES} samples)")
print_kv("Raw duration (median / mean)",
        f"{root_cause_df['raw_duration_s'].median():.3f}s / {root_cause_df['raw_duration_s'].mean():.3f}s")
print_kv("Resampled duration == raw duration",
        bool((root_cause_df['raw_duration_s'] - root_cause_df['resampled_duration_s']).abs().max() < 1e-6))
print_kv("VAD speech duration (median / mean)",
        f"{root_cause_df['speech_duration_s'].median():.3f}s / {root_cause_df['speech_duration_s'].mean():.3f}s")
print_kv("VAD fallback rate", f"{root_cause_df['fallback_used'].mean():.1%}")
print_kv("Median padding fraction of the 4s window",
        f"{root_cause_df['padding_fraction_of_window'].median():.1%}")

root_cause_path = config.METRICS_DIR / "vad_root_cause_summary.csv"
root_cause_df.to_csv(root_cause_path, index=False)
print_kv("Root-cause summary saved", root_cause_path)

root_cause_df[["Filename", "Speaker_ID", "raw_duration_s", "speech_duration_s",
              "speech_ratio", "padding_fraction_of_window", "fallback_used"]].head(10)


══════════════════════════════════════════════════════════════════════════════
  ROOT-CAUSE INVESTIGATION — MFCC TRAILING PADDING
══════════════════════════════════════════════════════════════════════════════
  Sample size ............................. 80
  Fixed analysis window ................... 4.0s (64000 samples)
  Raw duration (median / mean) ............ 2.108s / 2.880s
  Resampled duration == raw duration ...... True
  VAD speech duration (median / mean) ..... 0.604s / 1.130s
  VAD fallback rate ....................... 16.2%
  Median padding fraction of the 4s window . 84.9%
  Root-cause summary saved ................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\vad_root_cause_summary.csv


,Filename,Speaker_ID,raw_duration_s,speech_duration_s,speech_ratio,padding_fraction_of_window,fallback_used
0,M11_B1_CW13_M6.wav,M11,2.892063,0.284,0.098200,0.929,False
1,CF02_B2_CW85_M6.wav,CF02,1.877875,0.604,0.321640,0.849,False
2,M05_B3_UW97_M6.wav,M05,2.208312,0.188,0.085133,0.953,False
3,CM06_B2_UW74_M6.wav,CM06,1.647438,0.604,0.366630,0.849,False
4,CM01_B1_CW62_M6.wav,CM01,1.691250,0.476,0.281449,0.881,False
5,M10_B1_D2_M6.wav,M10,2.260063,0.444,0.196455,0.889,False
6,CF05_B3_LH_M6.wav,CF05,1.965750,0.796,0.404935,0.801,False
7,CM05_B3_UW10_M6.wav,CM05,2.009812,0.668,0.332369,0.833,False
8,CF03_B1_CW61_M6.wav,CF03,1.635063,0.572,0.349834,0.857,False
9,M08_B2_CW57_M6.wav,M08,2.428875,0.636,0.261850,0.841,False


In [11]:
# STAGE 10 - Is config.MAX_SAMPLES=64000 (400 MFCC frames) actually justified?
#
# Stage 9 measured this on an 80-file sample; here it's the full M6 manifest
# (reusing outputs/vad_stats.csv if Stage 9 / src.preprocessing.compute_vad_stats_batch
# already built it - this does not re-run VAD if so). Converts each
# utterance's VAD speech_duration_s into a valid MFCC frame count via
# src.preprocessing.mfcc_frame_count (the same formula the dataset, the
# Acoustic Pathway, and notebooks/02's VAD-validation figure all now share),
# then reports the distribution so max_frames=400 is a measured choice, not
# an assumption - see the module docstring in notebooks/02_feature_analysis.ipynb
# Stage 2 for how the 400-frame fixed window shows up as padding.
import numpy as np

from src.preprocessing import compute_vad_stats_batch, mfcc_frame_count

vad_stats_full = compute_vad_stats_batch(df_m6)
valid_frames_full = vad_stats_full["speech_duration_s"].apply(
    lambda s: mfcc_frame_count(int(round(s * config.TARGET_SR))))

percentiles = [0, 50, 75, 90, 95, 99, 100]
frame_distribution = {
    f"p{p}" if p not in (0, 100) else ("min" if p == 0 else "max"):
        int(np.percentile(valid_frames_full, p))
    for p in percentiles
}
frame_distribution["mean"] = float(valid_frames_full.mean())

current_max_frames = mfcc_frame_count(config.MAX_SAMPLES)
below_window = valid_frames_full < current_max_frames
at_window = valid_frames_full == current_max_frames
above_window = valid_frames_full > current_max_frames
padding_fractions = ((current_max_frames - valid_frames_full[below_window]) / current_max_frames)

print_header("Valid MFCC frame distribution (full M6 manifest)")
print_kv("Utterances", len(valid_frames_full))
print_kv("Minimum valid frames", frame_distribution["min"])
print_kv("Median valid frames (p50)", frame_distribution["p50"])
print_kv("Mean valid frames", f"{frame_distribution['mean']:.1f}")
print_kv("75th percentile", frame_distribution["p75"])
print_kv("90th percentile", frame_distribution["p90"])
print_kv("95th percentile", frame_distribution["p95"])
print_kv("99th percentile", frame_distribution["p99"])
print_kv("Maximum valid frames", frame_distribution["max"])
print_kv("Current config.MAX_SAMPLES frame count", current_max_frames)
print_kv("Utterances requiring padding (< fixed window)",
        f"{below_window.sum():,}/{len(valid_frames_full):,} ({below_window.mean():.2%})")
print_kv("Utterances exactly at fixed window", f"{at_window.sum():,}")
print_kv("Utterances exceeding fixed window (truncated)",
        f"{above_window.sum():,}/{len(valid_frames_full):,} ({above_window.mean():.2%})")
print_kv("Median padding fraction (padded utterances)", f"{padding_fractions.median():.2%}")
print_kv("Mean padding fraction (padded utterances)", f"{padding_fractions.mean():.2%}")

frame_distribution_path = config.METRICS_DIR / "mfcc_valid_frame_distribution.csv"
pd.Series(frame_distribution).to_frame("frames").to_csv(frame_distribution_path)
print_kv("Distribution saved", frame_distribution_path)

# NOTE: this is a REPORT, not an automatic change. config.MAX_SAMPLES stays at
# 4.0s / 400 frames here - shrinking it is a real architectural decision
# (every cached MFCC tensor, every checkpoint's Conv1d/attention shapes, and
# every already-run experiment's frame budget assumes it) that the team should
# make deliberately from the numbers above, not have silently changed by this
# investigation. If p99 is far below 400, a smaller max_frames (e.g. rounding
# p99 up to the nearest pooling-friendly multiple of 4, since AcousticPathway's
# two MaxPool1d(2) stages want an even frame count at each stage) would cut
# the median padding fraction dramatically without truncating almost any
# utterance's real speech.


─── VAD analysis — 21,420 utterances ─────────────────────────────────────────
  Computing VAD stats              100%|█████████████████████| 21420/21420 [09:35<00:00, 37.25utt/s]
  [ ✗ ] 18,501/21,420 utterances VAD-trimmed (2919 fell back to the original waveform)
  VAD stats cached ........................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\vad_stats.csv

══════════════════════════════════════════════════════════════════════════════
  VALID MFCC FRAME DISTRIBUTION (FULL M6 MANIFEST)
══════════════════════════════════════════════════════════════════════════════
  Utterances .............................. 21420
  Minimum valid frames .................... 19
  Median valid frames (p50) ............... 64
  Mean valid frames ....................... 105.4
  75th percentile ......................... 90
  90th percentile ......................... 222
  95th percentile ......................... 364
  99th percentile ....

In [12]:
# STAGE 11 - Primary severity protocol for the three-branch architecture:
# full-population Leave-One-Speaker-Out across all 15 dysarthric speakers
# (config.SEVERITY_PRIMARY_PROTOCOL == "full_loso"), NOT Stage 7's legacy
# 3-per-class balanced protocol (kept as a secondary sanity check only --
# see README "Evaluation protocol").
from src.splits import summarize_severity_loso_splits

summarize_severity_loso_splits(df_m6)


══════════════════════════════════════════════════════════════════════════════
  SEVERITY SPLITS (PRIMARY: FULL-POPULATION LEAVE-ONE-SPEAKER-OUT)
══════════════════════════════════════════════════════════════════════════════
  Total LOSO folds ........................ 15

─── Speakers per severity class ──────────────────────────────────────────────
  Very Low ................................ 4 speaker(s) — F03, M01, M04, M12
  Low ..................................... 3 speaker(s) — F02, M07, M16
  Mid ..................................... 3 speaker(s) — F04, M05, M11
  High .................................... 5 speaker(s) — F05, M08, M09, M10, M14

─── Example fold (F02 held out) ──────────────────────────────────────────────
  Train samples ........................... 10710
  Test samples ............................ 765


In [13]:
# STAGE 12 - Three-branch architecture bottleneck-dimension audit, derived
# from a real (untrained) model's tensors via one dummy forward pass --
# never hardcoded. See src.training.reporting.feature_audit.
from src.training.reporting import print_feature_audit

_ = print_feature_audit(num_classes=config.NUM_CLASSES["severity"])

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3760.20it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Wav2Vec2Model does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.



========== FEATURE AUDIT ==========

LEARNED BRANCH
Raw hidden representation: [B, T, 768]
Projected representation: [2, 128]
Learned representation dimensions: 128

SEGMENTAL BRANCH
Input feature channels: 43
Input tensor: [2, 43, 401]
Projected representation: [2, 64]
Segmental representation dimensions: 64
Segmental features (SHAP-surrogate table):
  1. jitter_local
  2. jitter_rap
  3. jitter_ppq5
  4. jitter_ddp
  5. shimmer_local
  6. shimmer_apq3
  7. shimmer_apq11
  8. shimmer_dda
  9. hnr_mean
  10. hnr_std
  11. hnr_min
  12. cpps
  13. f1_mean
  14. f2_mean
  15. f3_mean
  16. f1_std
  17. f2_std
  18. f3_std
  19. f2_f1_ratio
  20. voice_breaks

SUPRASEGMENTAL BRANCH
Input feature channels: 3
Input tensor: [2, 3, 401]
Projected representation: [2, 64]
Suprasegmental representation dimensions: 64
Suprasegmental features (SHAP-surrogate table):
  1. f0_mean
  2. f0_max
  3. f0_min
  4. f0_std
  5. f0_range
  6. intensity_mean
  7. intensity_max
  8. intensity_min
  9. intens

In [14]:
# STAGE 13 - Branch-specific framewise feature audit on a real utterance:
# segmental (43 channels: MFCC+delta+delta-delta + framewise formants + HNR)
# and suprasegmental (3 channels: F0 semitones + voicing mask + intensity),
# on their shared frame grid.
from src.preprocessing import extract_segmental_features_cached, extract_suprasegmental_features_cached

try:
    sample_row = df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].iloc[0]
    segmental_features = extract_segmental_features_cached(sample_row["Filepath"])
    supra_features = extract_suprasegmental_features_cached(sample_row["Filepath"])

    print_header("Framewise feature audit (sample utterance)")
    print_kv("Utterance", sample_row["Filename"])
    print_kv("Segmental tensor shape", tuple(segmental_features.shape))
    print_kv("Suprasegmental tensor shape", tuple(supra_features.shape))
    assert segmental_features.shape[0] == config.SEGMENTAL_CHANNELS == 43
    assert supra_features.shape[0] == config.SUPRA_CHANNELS == 3
    assert segmental_features.shape[-1] == supra_features.shape[-1], "branches must share one frame grid"
    print_kv("Frame grid agreement", "OK -- both branches share the same frame axis length")
except (FileNotFoundError, RuntimeError, OSError) as exc:
    print_kv("Skipped", f"no audio available in this checkout ({exc})")


══════════════════════════════════════════════════════════════════════════════
  FRAMEWISE FEATURE AUDIT (SAMPLE UTTERANCE)
══════════════════════════════════════════════════════════════════════════════
  Utterance ............................... F02_B1_C10_M6.wav
  Segmental tensor shape .................. (43, 401)
  Suprasegmental tensor shape ............. (3, 401)
  Frame grid agreement .................... OK -- both branches share the same frame axis length


In [15]:
# STAGE 14 - Missing/invalid feature statistics: what fraction of frames does
# the suprasegmental branch's voicing channel actually mark voiced, across a
# sample of dysarthric utterances? Low voiced-frame percentages are expected
# and clinically meaningful (dysarthric speech is often more aperiodic) --
# this is a diagnostic, not a bug indicator (see src.praat.extract_suprasegmental_sequence's
# never-fabricate-real-F0 contract).
try:
    import torch

    from src.praat import PITCH_FLOOR, PITCH_CEILING
    sample_dysarthric = df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].groupby(
        "Speaker_ID", group_keys=False).sample(n=5, random_state=config.DEFAULT_SEED)

    from src.preprocessing import load_and_preprocess_supra_cached
    voiced_fractions, f0_validity = [], []
    for row in sample_dysarthric.itertuples(index=False):
        supra = extract_suprasegmental_features_cached(row.Filepath)
        _, supra_valid_length = load_and_preprocess_supra_cached(row.Filepath)
        valid_frames = min(mfcc_frame_count(supra_valid_length), supra.shape[-1])
        voicing_channel = supra[1, :valid_frames]      # order: f0_semitones, voicing, intensity_db
        f0_channel = supra[0, :valid_frames]
        voiced_fractions.append(voicing_channel.mean().item())
        f0_validity.append((torch.isfinite(f0_channel[voicing_channel == 1]).all().item(),
                            (f0_channel[voicing_channel == 0] == 0).all().item()))

    print_header("F0 validity across a dysarthric-speaker sample")
    print_kv("Utterances sampled", len(sample_dysarthric))
    print_kv("Mean voiced-frame fraction", f"{sum(voiced_fractions) / len(voiced_fractions):.1%}"
            if voiced_fractions else "n/a")
    print_kv("Voiced fraction (min / median / p90 / max)",
             f"{np.min(voiced_fractions):.1%} / {np.median(voiced_fractions):.1%} / "
             f"{np.percentile(voiced_fractions, 90):.1%} / {np.max(voiced_fractions):.1%}")
    print_kv("Zero-voiced utterances", sum(v == 0 for v in voiced_fractions))
    print_kv("Extremely low voiced (<1%)", sum(v < .01 for v in voiced_fractions))
    print_kv("Praat F0 range", f"{PITCH_FLOOR:.0f}-{PITCH_CEILING:.0f} Hz")
    print_kv("Finite F0 wherever voiced", all(v[0] for v in f0_validity))
    print_kv("Unvoiced F0 represented as zero", all(v[1] for v in f0_validity))
except (FileNotFoundError, RuntimeError, OSError) as exc:
    print_kv("Skipped", f"no audio available in this checkout ({exc})")


══════════════════════════════════════════════════════════════════════════════
  F0 VALIDITY ACROSS A DYSARTHRIC-SPEAKER SAMPLE
══════════════════════════════════════════════════════════════════════════════
  Utterances sampled ...................... 75
  Mean voiced-frame fraction .............. 51.5%
  Voiced fraction (min / median / p90 / max) . 1.0% / 50.7% / 80.1% / 94.3%
  Zero-voiced utterances .................. 0
  Extremely low voiced (<1%) .............. 1
  Praat F0 range .......................... 75-600 Hz
  Finite F0 wherever voiced ............... True
  Unvoiced F0 represented as zero ......... True


In [16]:
# STAGE 15 - The two explicit preprocessing profiles side by side:
# speech-focused (config.VAD_SPEECH_PAD_MS, feeds Learned+Segmental) vs
# temporal-preserving (config.SUPRA_VAD_SPEECH_PAD_MS, feeds Suprasegmental
# only) -- the wider margin's effect is shown directly, not just documented.
from src.preprocessing import load_and_preprocess_cached, load_and_preprocess_supra_cached

try:
    profile_differences = []
    for row in sample_dysarthric.itertuples(index=False):
        _, speech_valid_length = load_and_preprocess_cached(row.Filepath)
        _, supra_valid_length = load_and_preprocess_supra_cached(row.Filepath)
        profile_differences.append(supra_valid_length - speech_valid_length)
    profile_differences = np.asarray(profile_differences)
    print_header("VAD profile comparison (stratified dysarthric sample)")
    print_kv("Speech-focused / temporal margins", f"{config.VAD_SPEECH_PAD_MS}ms / {config.SUPRA_VAD_SPEECH_PAD_MS}ms")
    print_kv("Utterances with additional temporal context",
             f"{(profile_differences > 0).sum()}/{len(profile_differences)} ({(profile_differences > 0).mean():.1%})")
    print_kv("Difference (mean / median / max) samples",
             f"{profile_differences.mean():.0f} / {np.median(profile_differences):.0f} / {profile_differences.max():.0f}")
except (FileNotFoundError, RuntimeError, OSError, NameError) as exc:
    print_kv("Skipped", f"no audio available in this checkout ({exc})")


══════════════════════════════════════════════════════════════════════════════
  VAD PROFILE COMPARISON (STRATIFIED DYSARTHRIC SAMPLE)
══════════════════════════════════════════════════════════════════════════════
  Speech-focused / temporal margins ....... 30ms / 150ms
  Utterances with additional temporal context . 57/75 (76.0%)
  Difference (mean / median / max) samples . 2918 / 3840 / 3840


In [17]:
# STAGE 16 - Final dataset sanity checks: the three-branch architecture's
# tensors must agree with config.py's frozen dimensions end to end, checked
# against a real (untrained) model rather than merely asserted in isolation.
import torch

from src.console import print_status
from src.models.gated_fusion import GatedFusionModel
from src.preprocessing import mfcc_frame_count

total_frames = mfcc_frame_count(config.MAX_SAMPLES)
dummy_model = GatedFusionModel(num_classes=config.NUM_CLASSES["severity"], num_speakers=2, use_lora=True)
dummy_model.eval()

dummy_waveform = torch.zeros(1, config.MAX_SAMPLES)
dummy_mfcc = torch.zeros(1, config.SEGMENTAL_CHANNELS, total_frames)
dummy_supra = torch.zeros(1, config.SUPRA_CHANNELS, total_frames)
dummy_mask = torch.ones(1, config.MAX_SAMPLES, dtype=torch.bool)
dummy_valid_frames = torch.full((1,), total_frames, dtype=torch.long)

with torch.no_grad():
    dummy_logits = dummy_model(waveform=dummy_waveform, mfcc=dummy_mfcc, attention_mask=dummy_mask,
                               supra=dummy_supra, supra_valid_frames=dummy_valid_frames)

assert dummy_logits.shape == (1, config.NUM_CLASSES["severity"])
assert torch.isfinite(dummy_logits).all()
print_status("Dataset -> three-branch architecture pipeline is shape-consistent end to end", ok=True)
print_status(f"Manifest saved at {config.MANIFEST_PATH} -- ready for notebooks/03_training.ipynb", ok=True)

# Separate backward-path sanity check. LayerDrop is deliberately disabled only
# for this one deterministic audit: in a normal training-mode single pass it
# may skip an encoder layer, making that layer's valid LoRA gradient absent.
gradient_model = GatedFusionModel(num_classes=config.NUM_CLASSES["severity"], num_speakers=2, use_lora=True).train()
gradient_backbone = gradient_model.deep_pathway.wav2vec.base_model.model
assert not gradient_backbone.config.apply_spec_augment
assert gradient_backbone.config.mask_time_prob == gradient_backbone.config.mask_feature_prob == 0.0
assert not hasattr(gradient_backbone, "masked_spec_embed")
original_layerdrop = gradient_backbone.config.layerdrop
gradient_backbone.config.layerdrop = 0.0
gradient_logits = gradient_model(
    waveform=torch.randn(1, config.MAX_SAMPLES),
    mfcc=torch.randn(1, config.SEGMENTAL_CHANNELS, total_frames),
    attention_mask=torch.ones(1, config.MAX_SAMPLES, dtype=torch.long),
    supra=torch.randn(1, config.SUPRA_CHANNELS, total_frames),
    supra_valid_frames=torch.full((1,), total_frames, dtype=torch.long))
gradient_logits.sum().backward()
lora_params = [(name, param) for name, param in gradient_model.named_parameters() if "lora_" in name]
assert lora_params and all(p.grad is not None and torch.isfinite(p.grad).all() for _, p in lora_params)
assert all(not p.requires_grad and p.grad is None for name, p in gradient_model.deep_pathway.named_parameters() if "lora_" not in name)
gradient_backbone.config.layerdrop = original_layerdrop
print_header("Wav2Vec2 + LoRA backward-path sanity check")
print_kv("Checkpoint", config.WAV2VEC_MODEL_NAME)
print_kv("SpecAugment", "disabled: checkpoint omits masked_spec_embed")
print_kv("Trainable LoRA tensors with finite gradients", f"{len(lora_params)}/{len(lora_params)}")
print_status("Frozen Wav2Vec2 base parameters have no gradients; LoRA path is live", ok=True)

# Speaker identity must be the unit of separation in every reported LOSO fold.
from src.splits import iter_loso_folds, iter_severity_loso_folds
for protocol, folds, expected in (("Detection LOSO", iter_loso_folds(df_m6), 28),
                                  ("Severity PRIMARY full LOSO", iter_severity_loso_folds(df_m6), 15)):
    audited = 0
    for held_out, train_df, test_df in folds:
        assert not (set(train_df.Speaker_ID) & set(test_df.Speaker_ID))
        assert set(test_df.Speaker_ID) == {held_out} and len(test_df) == config.WORDS_PER_SPEAKER
        audited += 1
    assert audited == expected
    print_status(f"{protocol}: {audited} folds, no speaker or held-out-utterance leakage", ok=True)

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 7427.11it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [ ✓ ] Dataset -> three-branch architecture pipeline is shape-consistent end to end
  [ ✓ ] Manifest saved at C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv -- ready for notebooks/03_training.ipynb


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 7612.76it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



══════════════════════════════════════════════════════════════════════════════
  WAV2VEC2 + LORA BACKWARD-PATH SANITY CHECK
══════════════════════════════════════════════════════════════════════════════
  Checkpoint .............................. facebook/wav2vec2-base-960h
  SpecAugment ............................. disabled: checkpoint omits masked_spec_embed
  Trainable LoRA tensors with finite gradients . 72/72
  [ ✓ ] Frozen Wav2Vec2 base parameters have no gradients; LoRA path is live
  [ ✓ ] Detection LOSO: 28 folds, no speaker or held-out-utterance leakage
  [ ✓ ] Severity PRIMARY full LOSO: 15 folds, no speaker or held-out-utterance leakage
